# Alpha Engine Research Operations v1.0

研究専用ノートです。既存のスコア・順位・採用銘柄・ウェイトを変更せず、同じ取得価格を保存して既存Live Screenerへ渡します。自動発注・売却・資産配分変更は行いません。

**各実行は新しいスナップショットを作ります。既存の運用ノートや本番実行フォルダは使用しません。** このノートのGoogle Colab実機・外部市場データ取得は別途検証が必要です。操作説明は `docs/alpha_engine_research_operations.md` を参照してください。


## 1. Google Driveを接続

研究用ファイルの保存先としてDriveを使用します。

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2. 研究設定（編集するのはこのセル）

`CODE_REF` は研究用ランナーを含む専用ブランチまたはコミット、`SOURCE_REF` は**既存エンジン本体**の比較対象です。実行時に両方をSHAへ解決して表示し、その後は取得先を更新しません。初期値のSOURCE_REFは既知の実行コミットです。エンジン本体がSOURCE_REFと一致しない場合、ランナーは停止します。

R100/N6が初期設定です。R60/N12では `RESIDUAL_RATIO=60`、`TOTAL_HOLDINGS=12` としてください。LOOKBACK_MONTHSは現行スクリーナーと同じ18か月を使用します。


In [ ]:
REPO_URL = 'https://github.com/avalon19920629-create/Alpha-Engine-backtest.git'
CODE_REF = 'codex/alpha-engine-research-operations-v1'
SOURCE_REF = '075f0985c76c2212552732534d7f3c741270a80d'

RESIDUAL_RATIO = 100
TOTAL_HOLDINGS = 6
LOOKBACK_MONTHS = 18
RESEARCH_OUTPUT_ROOT = '/content/drive/MyDrive/alpha_engine/research_runs'
CACHE_POLICY = 'new_snapshot'
RUN_MODE = 'research_only'

assert CACHE_POLICY == 'new_snapshot', 'v1.0は毎回新規スナップショットを保存します。'
assert RUN_MODE == 'research_only'
assert LOOKBACK_MONTHS == 18, '既存スクリーナーの取得期間を変更しません。'
assert SOURCE_REF and CODE_REF, 'Git参照を明示してください。'


## 3. 新しい作業ディレクトリへ取得・依存関係導入

このセルを再実行すると、`/content` に別の一時チェックアウトを作成します。既存チェックアウトへのpull、reset、上書きは行いません。研究専用依存関係をインストールします。GitHubから専用ブランチを取得できることが必要です。


In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_DIR = Path(tempfile.mkdtemp(prefix='alpha-engine-research-', dir='/content'))
subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_DIR)], check=True)

def resolve_local_ref(ref):
    # clone時点で取得できた参照だけを使う。参照を追加取得・更新しない。
    for candidate in (ref, f'refs/remotes/origin/{ref}'):
        result = subprocess.run(
            ['git', '-C', str(REPO_DIR), 'rev-parse', '--verify', '--end-of-options',
             f'{candidate}^{{commit}}'],
            text=True, capture_output=True,
        )
        if result.returncode == 0:
            return result.stdout.strip()
    raise RuntimeError(f'取得したリポジトリでGit参照を解決できません: {ref}')

CODE_SHA = resolve_local_ref(CODE_REF)
SOURCE_SHA = resolve_local_ref(SOURCE_REF)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', CODE_SHA], check=True)
# SOURCE_REFがブランチ名の場合も、clone済み参照をそのまま使用する。
RESOLVED_SOURCE_REF = SOURCE_REF
if subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', '--verify', '--end-of-options',
     f'{SOURCE_REF}^{{commit}}'], capture_output=True,
).returncode != 0:
    RESOLVED_SOURCE_REF = f'refs/remotes/origin/{SOURCE_REF}'
print('Research code SHA:', CODE_SHA)
print('Engine source SHA:', SOURCE_SHA)
print('New checkout:', REPO_DIR)
assert (REPO_DIR / 'alpha_engine_research_runner.py').is_file()
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-research.txt')],
    check=True,
)


## 4. 研究用ランナーを実行

価格取得はランナーの1回の呼び出しで行い、保存前の価格と同じDataFrameを既存スクリーナーへ渡します。キャッシュ再読込データでの照合は追加ダウンロードを行いません。

別Pythonプロセスで起動するため、セルの再実行やGit参照変更時に古いimportが残りません。失敗時には部分保存先を残し、成功とは表示しません。


In [ ]:
import json

RUN_RECEIPT = Path(tempfile.mkdtemp(
    prefix='alpha-engine-receipt-', dir=str(REPO_DIR.parent),
)) / 'result.json'
run_config = {
    'source_ref': RESOLVED_SOURCE_REF,
    'expected_source_sha': SOURCE_SHA,
    'residual_ratio': RESIDUAL_RATIO,
    'total_holdings': TOTAL_HOLDINGS,
    'lookback_months': LOOKBACK_MONTHS,
    'output_root': RESEARCH_OUTPUT_ROOT,
}
child_code = r"""
import json
import sys
from pathlib import Path
from alpha_engine_research_runner import run_research

receipt = Path(sys.argv[2])
try:
    run_dir = run_research(**json.loads(sys.argv[1]))
except Exception as exc:
    receipt.write_text(json.dumps({
        'call_status': 'failed',
        'run_dir': str(getattr(exc, 'run_dir', '') or ''),
        'error': str(exc),
    }, ensure_ascii=False), encoding='utf-8')
    raise
else:
    receipt.write_text(json.dumps({
        'call_status': 'returned', 'run_dir': str(run_dir),
    }), encoding='utf-8')
"""
result = subprocess.run(
    [sys.executable, '-c', child_code, json.dumps(run_config), str(RUN_RECEIPT)],
    cwd=REPO_DIR,
)
receipt = json.loads(RUN_RECEIPT.read_text()) if RUN_RECEIPT.exists() else {
    'call_status': 'failed', 'run_dir': '', 'error': '結果通知を作成できませんでした。上のログを確認してください。',
}
RUN_DIR = Path(receipt['run_dir']) if receipt.get('run_dir') else None
print('Process exit code:', result.returncode)
print('Run receipt:', receipt)
if result.returncode:
    print('実行は失敗しました。次のセルで部分成果物と検査結果を確認してください。')


## 5. 結果・検査・価格キャッシュを確認

終了コードだけでなく、実行メタデータと検査結果を確認します。表示されない項目や欠損成果物を検証成功とは扱いません。

In [ ]:
import hashlib
import pandas as pd
from IPython.display import JSON, Markdown, display

if RUN_DIR is None:
    raise RuntimeError('保存先が不明です。実行セルのエラーを確認してください。')
meta_path = RUN_DIR / 'research_run_metadata.json'
if meta_path.is_file():
    run_meta = json.loads(meta_path.read_text(encoding='utf-8'))
    keys = ('status', 'validation_status', 'run_started_at', 'run_finished_at',
            'source_ref', 'git_commit_sha', 'observed_start', 'observed_end', 'price_cache_sha256')
    display(JSON({key: run_meta.get(key) for key in keys}))
    if run_meta.get('status') != 'completed' or run_meta.get('validation_status') != 'passed':
        print('未完了または検査失敗です。完成済みデータセットとして扱わないでください。')
for relative in ('research_run_metadata.json', 'audit/validation_report.json',
                 'price_data/price_cache_metadata.json', 'audit/file_manifest.json'):
    path = RUN_DIR / relative
    display(Markdown(f'**{relative}**'))
    if path.is_file():
        display(JSON(json.loads(path.read_text(encoding='utf-8'))))
    else:
        print('未作成:', path)

for relative in ('price_data/price_cache.parquet', 'price_data/benchmark_prices.parquet'):
    path = RUN_DIR / relative
    if path.is_file():
        frame = pd.read_parquet(path)
        print(relative, 'shape=', frame.shape,
              'observed=', frame.index.min(), 'to', frame.index.max(),
              'sha256=', hashlib.sha256(path.read_bytes()).hexdigest())

selected_path = RUN_DIR / 'engine_outputs/selected_tickers.csv'
if selected_path.is_file():
    display(pd.read_csv(selected_path))
quality_path = RUN_DIR / 'price_data/price_data_quality.csv'
if quality_path.is_file():
    display(pd.read_csv(quality_path))


## 6. 保存先と次回の実行

Driveの研究用フォルダで以下を確認してください。`research_run_metadata.json` の実行・検査ステータスが成功で、`audit/validation_report.json` に失敗がないことが必要です。失敗・未完了のフォルダは解析用の正常データセットと混同しないでください。

同じ条件で再実行する場合はセル4以降を実行します。新しい価格取得は別スナップショットです。後日修正された調整後価格を過去のキャッシュと同一とはみなしません。


In [ ]:
print('Research output root:', RESEARCH_OUTPUT_ROOT)
print('This run:', str(RUN_DIR) if RUN_DIR is not None else '(run directory unavailable)')
print('Engine source ref / SHA:', SOURCE_REF, SOURCE_SHA)
print('Research code ref / SHA:', CODE_REF, CODE_SHA)
print('Driveで上の保存先を開き、price_data・engine_outputs・auditを確認してください。')
print('研究用データは現在ユニバースと現在取得した調整後価格です。過去のPITユニバースは復元しません。')
